In [ ]:
# Importing libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
input_path = r"input.csv"  # Change this to your input file path
df = pd.read_csv(input_path, delimiter = ";") 
df.head()

,TRANSPORTER_TYPE,ROUTE,SERVICE_NUMBER,DEPARTURE_DATE,ARRIVAL_DATE,DAY_X,CAPTURE_DATE,SERVICE_ORIGIN,SERVICE_DESTINATION,DEPARTURE_TIME,...,FCST_REVENUE_EXCL_TAX,FCST_PRcphCTION,FCST_OBSERVATION,OPTIM_BOOKINGS,OPTIM_REVENUE_INCL_TAX,OPTIM_REVENUE_EXCL_TAX,UNCONSTRAINED_FORECAST_BOOKINGS,UNCONSTRAINED_FORECAST_REVENUE,UNCONSTRAINED_DEMAND_BOOKINGS,UNCONSTRAINED_DEMAND_REVENUE
0,AW,cdg-osl,AW7641,25/11/2024,25/11/2024,-1,25/11/2024,osl,cdg,09:45,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AW,cdg-osl,AW7647,25/11/2024,25/11/2024,-1,25/11/2024,osl,cdg,15:00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AW,cdg-tll,AW3280,26/11/2024,26/11/2024,-1,25/11/2024,cdg,tll,07:15,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,AW,cdg-tll,AW3281,26/11/2024,26/11/2024,-1,25/11/2024,tll,cdg,13:55,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,AW,cdg-osl,AW7646,26/11/2024,26/11/2024,-1,25/11/2024,cdg,osl,13:00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [163]:
df.columns

Index(['TRANSPORTER_TYPE', 'ROUTE', 'SERVICE_NUMBER', 'DEPARTURE_DATE',
       'ARRIVAL_DATE', 'DAY_X', 'CAPTURE_DATE', 'SERVICE_ORIGIN',
       'SERVICE_DESTINATION', 'DEPARTURE_TIME', 'ARRIVAL_TIME', 'STATUS',
       'CABIN', 'CAPACITY', 'LID', 'CURRENT_BUCKET', 'CURRENT_BUCKET_PRICE',
       'BOOKINGS', 'REVENUE_INCL_TAX', 'REVENUE_EXCL_TAX', 'MAXLEG_LF',
       'MAXLEG_BOOKINGS', 'YIELD_OBJ', 'BUDGET_OBJ', 'OPTIM_STATUS',
       'FCST_BOOKINGS', 'FCST_REVENUE_INCL_TAX', 'FCST_REVENUE_EXCL_TAX',
       'FCST_PRcphCTION', 'FCST_OBSERVATION', 'OPTIM_BOOKINGS',
       'OPTIM_REVENUE_INCL_TAX', 'OPTIM_REVENUE_EXCL_TAX',
       'UNCONSTRAINED_FORECAST_BOOKINGS', 'UNCONSTRAINED_FORECAST_REVENUE',
       'UNCONSTRAINED_DEMAND_BOOKINGS', 'UNCONSTRAINED_DEMAND_REVENUE'],
      dtype='object')

In [ ]:
df["ROUTE"] = (df["SERVICE_ORIGIN"] + " - " + df["SERVICE_DESTINATION"]).str.upper() # Creating ROUTE column (Initial Route Column did not consider the order of origin and desination)

df['ARRIVAL'] = pd.to_datetime(df['ARRIVAL_DATE'] + ' ' + df['ARRIVAL_TIME']) # Creating <datetime> ARRIVAL

df['DEPARTURE'] = pd.to_datetime(df['DEPARTURE_DATE'] + ' ' + df['DEPARTURE_TIME']) # Creating <datetime> DEPARTURE

df['CAPTURE'] = pd.to_datetime(df['CAPTURE_DATE']) # Creating <date> CAPTURE

df = df[['SERVICE_NUMBER', 'ROUTE', 'DEPARTURE', 'ARRIVAL', 'CAPTURE', 'DAY_X',   # Getting rid of forecast and optimization columns
        'STATUS', 'CABIN', 'CAPACITY', 'LID', 'CURRENT_BUCKET', 'CURRENT_BUCKET_PRICE',
       'BOOKINGS', 'REVENUE_INCL_TAX', 'REVENUE_EXCL_TAX', 'MAXLEG_LF',
       'MAXLEG_BOOKINGS', 'YIELD_OBJ', 'BUDGET_OBJ', 'OPTIM_STATUS']]
        
df = df.sort_values(by='DEPARTURE', ascending=True) 

C:\Users\PRASHANT\AppData\Local\Temp\ipykernel_34776\3426027951.py:3: UserWarning: Parsing dates in %d/%m/%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df['ARRIVAL'] = pd.to_datetime(df['ARRIVAL_DATE'] + ' ' + df['ARRIVAL_TIME'])
C:\Users\PRASHANT\AppData\Local\Temp\ipykernel_34776\3426027951.py:5: UserWarning: Parsing dates in %d/%m/%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df['DEPARTURE'] = pd.to_datetime(df['DEPARTURE_DATE'] + ' ' + df['DEPARTURE_TIME'])
C:\Users\PRASHANT\AppData\Local\Temp\ipykernel_34776\3426027951.py:7: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df['CAPTURE'] = pd.to_datetime(df['CAPTURE_DATE'])


In [165]:
df.head()

,SERVICE_NUMBER,ROUTE,DEPARTURE,ARRIVAL,CAPTURE,DAY_X,STATUS,CABIN,CAPACITY,LID,CURRENT_BUCKET,CURRENT_BUCKET_PRICE,BOOKINGS,REVENUE_INCL_TAX,REVENUE_EXCL_TAX,MAXLEG_LF,MAXLEG_BOOKINGS,YIELD_OBJ,BUDGET_OBJ,OPTIM_STATUS
11436,AW3121,ATH - CDG,2023-10-01 05:55:00,2023-10-01 10:25:00,2023-10-03,1,OPEN,2,189,191,B,285.0,178,"24063,1","20808,07","0,93",178,NaN,NaN,DEACTIVATED
10990,AW4781,VIE - CDG,2023-10-01 06:00:00,2023-10-01 08:00:00,2023-10-02,1,OPEN,2,189,192,Q,150.0,175,14043,"10826,5","0,91",175,NaN,NaN,DEACTIVATED
13972,AW7661,OSL - CDG,2023-10-01 06:00:00,2023-10-01 09:30:00,2023-10-03,1,OPEN,2,189,193,S,284.0,188,"13852,75","9902,57","0,97",188,NaN,NaN,DEACTIVATED
13549,AW7663,OSL - CDG,2023-10-01 06:30:00,2023-10-01 10:00:00,2023-10-03,1,OPEN,2,189,193,S,284.0,187,"13005,64","9009,75","0,97",187,NaN,NaN,DEACTIVATED
17106,AW3953,ZAG - CDG,2023-10-01 06:30:00,2023-10-01 08:40:00,2023-10-02,1,OPEN,2,189,193,X,60.0,193,"15092,32","9165,1",1,193,NaN,NaN,DEACTIVATED


In [166]:
df.count()

SERVICE_NUMBER          17699
ROUTE                   17699
DEPARTURE               17699
ARRIVAL                 17699
CAPTURE                 17699
DAY_X                   17699
STATUS                  17699
CABIN                   17699
CAPACITY                17699
LID                     17699
CURRENT_BUCKET          16139
CURRENT_BUCKET_PRICE    16139
BOOKINGS                17699
REVENUE_INCL_TAX        17699
REVENUE_EXCL_TAX        17699
MAXLEG_LF               17609
MAXLEG_BOOKINGS         17699
YIELD_OBJ                   0
BUDGET_OBJ                  0
OPTIM_STATUS            17699
dtype: int64

In [ ]:
df.drop(["YIELD_OBJ", "BUDGET_OBJ"], axis=1, inplace=True) # Dropping columns that are empty

In [ ]:
print(df['DAY_X'].value_counts()) # Checking the distribution of DAY_X

'''
It can be seen that most of the data is post departure 
Post departure data count: 10840 (DAY_X = 1; i.e; data is captured 1 day after departure)
Pre departure data count:  6859 (DAY_X < 1; i.e; data is captured "DAY_X" days before departure)
'''


DAY_X
 1      10840
-275       30
-233       30
-240       30
-247       30
        ...  
-71        17
-43        17
-64        17
-50        17
-287        5
Name: count, Length: 289, dtype: int64


In [ ]:
numeric_columns = ['REVENUE_INCL_TAX', 'REVENUE_EXCL_TAX', 'MAXLEG_LF'] # These columns are string because of the comma, which is actually a decimel
for col in numeric_columns:
    if df[col].dtype == 'object':  
        df[col] = df[col].str.replace(',', '.').astype(float)

In [170]:
df.head()

,SERVICE_NUMBER,ROUTE,DEPARTURE,ARRIVAL,CAPTURE,DAY_X,STATUS,CABIN,CAPACITY,LID,CURRENT_BUCKET,CURRENT_BUCKET_PRICE,BOOKINGS,REVENUE_INCL_TAX,REVENUE_EXCL_TAX,MAXLEG_LF,MAXLEG_BOOKINGS,OPTIM_STATUS
11436,AW3121,ATH - CDG,2023-10-01 05:55:00,2023-10-01 10:25:00,2023-10-03,1,OPEN,2,189,191,B,285.0,178,24063.10,20808.07,0.93,178,DEACTIVATED
10990,AW4781,VIE - CDG,2023-10-01 06:00:00,2023-10-01 08:00:00,2023-10-02,1,OPEN,2,189,192,Q,150.0,175,14043.00,10826.50,0.91,175,DEACTIVATED
13972,AW7661,OSL - CDG,2023-10-01 06:00:00,2023-10-01 09:30:00,2023-10-03,1,OPEN,2,189,193,S,284.0,188,13852.75,9902.57,0.97,188,DEACTIVATED
13549,AW7663,OSL - CDG,2023-10-01 06:30:00,2023-10-01 10:00:00,2023-10-03,1,OPEN,2,189,193,S,284.0,187,13005.64,9009.75,0.97,187,DEACTIVATED
17106,AW3953,ZAG - CDG,2023-10-01 06:30:00,2023-10-01 08:40:00,2023-10-02,1,OPEN,2,189,193,X,60.0,193,15092.32,9165.10,1.00,193,DEACTIVATED


In [175]:
dfp = df[df["DAY_X"] > 0]
dff = df[df["DAY_X"] <=0]

In [ ]:
# No. of flights and date range for DAY_X > 0 and DAY_X <= 0
print("Number of flights with DAY_X > 0:", dfp.shape[0])
print("Date range for DAY_X > 0:")
print(dfp["DEPARTURE"].min(), " - ", dfp["DEPARTURE"].max())
print("Number of flights with DAY_X <= 0:", dff.shape[0])
print("Date range for DAY_X <= 0:")
print(dff["DEPARTURE"].min(),  " - ", dff["DEPARTURE"].max())


Number of flights with DAY_X > 0: 10840
Date range for DAY_X > 0:
2023-10-01 05:55:00  -  2024-11-24 20:45:00
Number of flights with DAY_X <= 0: 6859
Date range for DAY_X <= 0:
2024-11-24 10:35:00  -  2025-09-07 21:25:00


In [179]:
dfp.reset_index(drop=True, inplace=True)
dfp.head()

,SERVICE_NUMBER,ROUTE,DEPARTURE,ARRIVAL,CAPTURE,DAY_X,STATUS,CABIN,CAPACITY,LID,...,BOOKINGS,REVENUE_INCL_TAX,REVENUE_EXCL_TAX,MAXLEG_LF,MAXLEG_BOOKINGS,OPTIM_STATUS,DISTANCE,AIRPORT_TAX,COST,PROFIT
0,AW3121,ATH - CDG,2023-10-01 05:55:00,2023-10-01 10:25:00,2023-10-03,1,OPEN,2,189,191,...,178,24063.10,20808.07,0.93,178,DEACTIVATED,2100,18.39,11190.42,12872.68
1,AW4781,VIE - CDG,2023-10-01 06:00:00,2023-10-01 08:00:00,2023-10-02,1,OPEN,2,189,192,...,175,14043.00,10826.50,0.91,175,DEACTIVATED,1035,18.80,7191.95,6851.05
2,AW7661,OSL - CDG,2023-10-01 06:00:00,2023-10-01 09:30:00,2023-10-03,1,OPEN,2,189,193,...,188,13852.75,9902.57,0.97,188,DEACTIVATED,1350,22.61,9340.18,4512.57
3,AW7663,OSL - CDG,2023-10-01 06:30:00,2023-10-01 10:00:00,2023-10-03,1,OPEN,2,189,193,...,187,13005.64,9009.75,0.97,187,DEACTIVATED,1350,22.61,9317.57,3688.07
4,AW3953,ZAG - CDG,2023-10-01 06:30:00,2023-10-01 08:40:00,2023-10-02,1,OPEN,2,189,193,...,193,15092.32,9165.10,1.00,193,DEACTIVATED,1080,31.56,10162.68,4929.64


In [180]:
dff.reset_index(drop=True, inplace=True)
dff.head()

,SERVICE_NUMBER,ROUTE,DEPARTURE,ARRIVAL,CAPTURE,DAY_X,STATUS,CABIN,CAPACITY,LID,...,BOOKINGS,REVENUE_INCL_TAX,REVENUE_EXCL_TAX,MAXLEG_LF,MAXLEG_BOOKINGS,OPTIM_STATUS,DISTANCE,AIRPORT_TAX,COST,PROFIT
0,AW4671,BUD - CDG,2024-11-24 10:35:00,2024-11-24 15:40:00,2024-11-25,0,OPEN,2,186,188,...,155,10393.64,8871.54,0.82,155,DEACTIVATED,1250,9.82,6234.60,4159.04
1,AW7643,OSL - CDG,2024-11-24 12:55:00,2024-11-24 16:30:00,2024-11-25,0,OPEN,2,189,193,...,190,20031.00,15760.09,0.98,190,ACTIVATED,1350,22.61,9385.40,10645.60
2,AW7647,OSL - CDG,2024-11-24 13:55:00,2024-11-24 17:30:00,2024-11-25,0,OPEN,2,189,193,...,188,20181.22,15938.34,0.97,188,ACTIVATED,1350,22.61,9340.18,10841.04
3,AW7645,OSL - CDG,2024-11-24 15:40:00,2024-11-24 19:15:00,2024-11-25,0,CANCELLED,2,189,189,...,0,0.00,0.00,0.00,0,UNAVAILABLE,1350,22.61,0.00,0.00
4,AW7649,OSL - CDG,2024-11-24 18:40:00,2024-11-24 22:15:00,2024-11-25,0,OPEN,2,186,193,...,190,22897.70,18603.75,0.98,190,ACTIVATED,1350,22.61,9385.40,13512.30


In [ ]:
cleaned_output_path = r"AirWiremind_Cleaned.csv"  # Save the cleaned DataFrame
df.to_csv(cleaned_output_path, index=False)

In [ ]:
post_departure_output_path = r"AirWiremind_Completed.csv"  # Save the post-departure DataFrame
dfp.to_csv(post_departure_output_path)

pre_departure_output_path = r"AirWiremind_Future.csv"  # Save the pre-departure DataFrame
dff.to_csv(pre_departure_output_path)

''' 
Note: Do not change the CSV filename, i.e., "AirWiremind_Cleaned.csv", "AirWiremind_Completed.csv" and "AirWiremind_Future.csv"
      as these are used in the next steps (Power BI) of the project.
      However, you can change the path where these files are saved.

      To see the dashboard, you can open the Power BI file "AirWiremind_Dashboard.pbix" in Power BI Desktop.
      You will notice that the dashboard cannot be opened until you have the data files in the same directory as the Power BI file (Power Query).
      Open the Power Query Editor in Power BI and change the path to the directory (in Applied Steps - > Source, on the right side of the screen)
      where you saved the two data files "AirWiremind_Completed.csv" and "AirWiremind_Future.csv" .
'''

# Visualization could have been done here, but Power BI would give a dynamic and better visualization experience.